#### Setup LangChain

pip install -U langchain langchain_community langchain-openai

In [1]:
import langchain
langchain.__version__

'1.2.17'


#### Credentials

In [2]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [3]:
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

os.environ["LANGSMITH_TRACING"] = "true"

#### Instantiation

In [2]:
import os
from langchain_openai import ChatOpenAI
from com.example.ai.LLMManager import LLMManager
model = LLMManager.get_model()


#### Invocation : Create Prompt

In [3]:
messages = [
    ("system", "You are a helpful assistant that translates English to Hindi. Translate the user sentence."),
    ("human", "I love programming."),
]

ai_msg = model.invoke("translate I love programming to Hindi")
ai_msg

AIMessage(content='The most natural and commonly used translation for "I love programming" in Hindi is:\n\n**मुझे कोडिंग से प्यार है।**\n(Mujhe coding se pyaar hai.)\n\n***\n\n### 📚 Alternative Translations (Depending on Context)\n\nIf you want to sound more formal or specific:\n\n**1. Formal/Academic (Focus on the act):**\n* **मुझे प्रोग्रामिंग करना पसंद है।**\n  (Mujhe programming karna pasand hai.)\n  *(Literally: I like to do programming.)*\n\n**2. Very Intense Love/Passion (Focus on passion):**\n* **मुझे कोडिंग का बहुत शौक़ है।** (More emphasizes interest/passion)\n  (Mujhe coding ka bahut shauk hai.)\n\n***\n\n### 💡 Breakdown of the Recommended Translation:\n\n* **मुझे** (Mujhe) - To me / I (object form, used frequently when expressing feelings)\n* **कोडिंग** (Coding) - Programming (The Hindi phonetic transliteration of the English word, which is widely accepted.)\n* **से** (Se) - From / With (Used here to connect the feeling of love *from* programming)\n* **प्यार है** (Pyaar hai

```json
AIMessage(
    content="J'adore la programmation.",
    response_metadata={
        "token_usage": {
            "completion_tokens": 5,
            "prompt_tokens": 31,
            "total_tokens": 36,
        },
        "model_name": "gpt-4o",
        "system_fingerprint": "fp_43dfabdef1",
        "finish_reason": "stop",
        "logprobs": None,
    },
    id="run-012cffe2-5d3d-424d-83b5-51c6d4a593d1-0",
    usage_metadata={"input_tokens": 31, "output_tokens": 5, "total_tokens": 36},
)
```

In [4]:
print(ai_msg.text)

The most natural and commonly used translation for "I love programming" in Hindi is:

**मुझे कोडिंग से प्यार है।**
(Mujhe coding se pyaar hai.)

***

### 📚 Alternative Translations (Depending on Context)

If you want to sound more formal or specific:

**1. Formal/Academic (Focus on the act):**
* **मुझे प्रोग्रामिंग करना पसंद है।**
  (Mujhe programming karna pasand hai.)
  *(Literally: I like to do programming.)*

**2. Very Intense Love/Passion (Focus on passion):**
* **मुझे कोडिंग का बहुत शौक़ है।** (More emphasizes interest/passion)
  (Mujhe coding ka bahut shauk hai.)

***

### 💡 Breakdown of the Recommended Translation:

* **मुझे** (Mujhe) - To me / I (object form, used frequently when expressing feelings)
* **कोडिंग** (Coding) - Programming (The Hindi phonetic transliteration of the English word, which is widely accepted.)
* **से** (Se) - From / With (Used here to connect the feeling of love *from* programming)
* **प्यार है** (Pyaar hai) - Is love (Love)


### Stream

In [5]:
for chunk in model.stream(messages):
    chunk
    print(chunk.text, end="")

मुझे प्रोग्रामिंग पसंद है।

To collect the full message, you can concatenate the chunks:

In [6]:
stream = model.stream(messages)
full = next(stream)
for chunk in stream:
    full += chunk

In [ ]:
full = AIMessageChunk(
    content="J'adore la programmation.",
    response_metadata={"finish_reason": "stop"},
    id="run-bf917526-7f58-4683-84f7-36a6b671d140",
)

NameError: name 'AIMessage' is not defined

`Async` : Asynchronous equivalents of invoke, stream, and batch are also available:

In [ ]:
# Invoke
await model.ainvoke(messages)

# Stream
async for chunk in (await model.astream(messages))

# Batch
await model.abatch([messages])

Tool calling

In [10]:
from pydantic import BaseModel, Field

class GetWeather(BaseModel):
    '''Get the current weather in a given location'''
    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")

class GetPopulation(BaseModel):
    '''Get the current population for a given location'''
    population: float = Field(..., description="The city and population, e.g. San Francisco, 10.55")

model_with_tools = model.bind_tools(
    [GetWeather, GetPopulation]
    # strict = True  # Enforce tool args schema is respected
)

ai_msg = model_with_tools.invoke(
    "Which city is hotter today and which is bigger: LA or NY?"
)

ai_msg.tool_calls[
    {
        "name": "GetWeather",
        "args": {"location": "Los Angeles, CA"},
        "id": "call_6XswGD5Pqk8Tt5atYr7tfenU",
    },
    {
        "name": "GetWeather",
        "args": {"location": "New York, NY"},
        "id": "call_ZVL15vA8Y7kXqOy3dtmQgeCi",
    },
    {
        "name": "GetPopulation",
        "args": {"location": "Los Angeles, CA"},
        "id": "call_49CFW8zqC9W7mh7hbMLSIrXw",
    },
    {
        "name": "GetPopulation",
        "args": {"location": "New York, NY"},
        "id": "call_6ghfKxV264jEfe1mRIkS3PE7",
    },
]

TypeError: list indices must be integers or slices, not tuple

Parallel tool calls

In [11]:
ai_msg = model_with_tools.invoke(
    "What is the weather in LA and NY?", parallel_tool_calls=False
)
ai_msg.tool_calls[
    {
        "name": "GetWeather",
        "args": {"location": "Los Angeles, CA"},
        "id": "call_4OoY0ZR99iEvC7fevsH8Uhtz",
    }
]

TypeError: list indices must be integers or slices, not dict

Built-in (server-side) tools

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="...", output_version="responses/v1")

tool = {"type": "web_search"}
model_with_tools = model.bind_tools([tool])

response = model_with_tools.invoke("What was a positive news story from today?")
response.content[
    {
        "type": "text",
        "text": "Today, a heartwarming story emerged from ...",
        "annotations": [
            {
                "end_index": 778,
                "start_index": 682,
                "title": "Title of story",
                "type": "url_citation",
                "url": "<url of story>",
            }
        ],
    }
]